In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client()
model = "gemini-3.5-flash-lite"


In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "parts": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "model", "parts": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, tools=None):
    config = types.GenerateContentConfig(
        max_output_tokens=1000,
        temperature=temperature,
    )
    if system:
        config.system_instruction = system
    if tools:
        config.tools = tools

    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=config,
    )
    return response.text


def create_tool_chat(tools, system=None, temperature=1.0):
    """Creates an interactive chat session with Automatic Function Calling (AFC) enabled."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        tools=tools,
    )
    if system:
        config.system_instruction = system
    return client.chats.create(model=model, config=config)


In [7]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str: str, duration: int = 0, unit: str = "days", input_format: str = "%Y-%m-%d"
) -> str:
    """Adds a specified duration to a datetime string and returns the resulting datetime formatted."""
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content: str, timestamp: str) -> str:
    """Creates a timed reminder that will notify the user at the specified time with content."""
    print(f"----\nSetting reminder for {timestamp}:\n{content}\n----\n")
    return f"Reminder successfully set for {timestamp}: {content}"


def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """Returns the current date and time formatted."""
    return datetime.now().strftime(date_format)


# Gemini Function Declarations / Schemas (OpenAPI parameters schema)
add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format.",
    "parameters": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added.",
            },
            "duration": {
                "type": "integer",
                "description": "The amount of time to add to the datetime. Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "Unit of time: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing datetime_str. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content.",
    "parameters": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

# In the Google Gemini SDK, you can pass Python functions directly in the tools list!
tools = [get_current_datetime, add_duration_to_datetime, set_reminder]


In [8]:
# Test the helper function directly
current_time = get_current_datetime("%Y-%m-%d %H:%M:%S")
print("Current datetime:", current_time)

get_current_datetime_schema = {
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted as a string using Python strftime directives.",
  "parameters": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "The format string for the date and time (e.g., '%Y-%m-%d %H:%M:%S', '%d-%m').",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
}




Current datetime: 2026-10-31 05:32:32


In [18]:
messages = []
add_user_message(messages, "what is the exact time , formatted as HH:MM:SS?")

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[{"function_declarations": [get_current_datetime_schema]}]
    ),
)

messages.append({
    "role": "model",  # Use 'model', not 'assistant'
    "parts": response.candidates[0].content.parts,
})

messages

[{'role': 'user',
  'parts': [{'text': 'what is the exact time , formatted as HH:MM:SS?'}]},
 {'role': 'model',
  'parts': [Part(
     function_call=FunctionCall(
       args={
         'date_format': '%H:%M:%S'
       },
       id='call_448616',
       name='get_current_datetime'
     ),
     thought_signature=b'\x12`\n^\x01i\x14}\x131\xff\x00\x0f\x1e\xc4~\xdf\x8f\xd5\xb4`I{\r\xa9\xd2\xb0;\xaa\x87(\xc7J?j\x84s\\a\xb1\x9d!\xe0\xa5\xb2]0\x1a\\\xcdh\x00\xcd\xf2\xb3\xf7e\xa4r\x7f\xfd\xf3\xdc=\x83\x1dU\xde,k\x10\x10\x84\xb8\x90d\xe8\x13\xbe\x12\x9e\x89\x98j,L.j<\xe2lh\x9a\x9d\x1fQn\xf0'
   )]}]

In [ ]:
# 1. Execute the tool locally with the model's requested arguments
tool_call = response.function_calls[0]
tool_result = get_current_datetime(**tool_call.args)
print(f"Tool executed locally: {tool_result}")

# 2. Append the tool result to the conversation
messages.append(
    types.Content(
        role="user",
        parts=[
            types.Part.from_function_response(
                name=tool_call.name,
                response={"result": tool_result},
            )
        ],
    )
)

# 3. Call Gemini again with the tool result so it answers the question
final_response = client.models.generate_content(
    model=model,
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[{"function_declarations": [get_current_datetime_schema]}]
    ),
)

print(f"\nGemini: {final_response.text}")


In [ ]:
# Automatic Tool Calling Example with Gemini
# Gemini automatically invokes the Python functions and returns the final answer!

chat_session = create_tool_chat(
    tools=tools,
    system="You are a helpful personal assistant with access to datetime and reminder tools.",
)

prompt = "what is the exact time , formatted as HH:MM:SS?"
print(f"User: {prompt}\n")

response = chat_session.send_message(prompt)
print(f"Assistant: {response.text}")
